<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/CIS3120-BMWB/blob/main/CIS3120_Module15_PressRelease2Plot_EXTENDED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 15 — Press Release to Plot - Extended
## Stepwise Student Walkthrough

**Course:** CIS 3120 — Programming for Analytics
**Pipeline:** SEC EDGAR full-text search → Anthropic API extraction → folium map
**Companion to instructor notebook:** Module 15 INSTRUCTOR

This notebook is a guided, step-by-step construction of the same pipeline covered in the instructor notebook. Where the instructor notebook presents each exercise as a single function to write and then reveals the canonical solution, this walkthrough decomposes each exercise into a sequence of small, explained steps. Each step is preceded by a short text block that explains what the next code cell will do and why it is the correct next piece of the solution.

The three exercises walked through here are:

1. **Exercise S1** — Constructing a paginated EDGAR query function.
2. **Exercise S3** — Authoring an extraction system prompt for Claude.
3. **Exercise S5** — Building the folium marker loop that renders the map.



<i>A copy of this notebook is available at [bit.ly/cis3120_module15_x](https://bit.ly/cis3120_module15_x)

## Setup

The pipeline depends on four external libraries:

- `requests` — issues HTTP calls to SEC EDGAR and to OpenStreetMap
- `anthropic` — official Python SDK for the Claude API
- `folium` — Python wrapper that produces interactive Leaflet maps
- `beautifulsoup4` — parses HTML content from EDGAR exhibits

Of these, only `anthropic` and `folium` are guaranteed to require installation in a fresh Colab runtime.

In [ ]:
!pip install -q anthropic folium beautifulsoup4 requests

### Storing the API key in Colab Secrets

Anthropic API keys are sensitive credentials. Pasting one directly into a notebook cell creates real risks: the key becomes part of the notebook file, gets committed to GitHub, appears in screenshots shared during help sessions, and remains visible to anyone who opens the notebook later.

The standard solution in Colab is the Secrets manager (key icon in the left sidebar). Stored secrets are accessible only to your account and only when the notebook is run interactively. The pattern below reads the secret named `ANTHROPIC_API_KEY` and places it into the environment variable that the Anthropic SDK looks for automatically.

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
EMAIL_ADDRESS = userdata.get("EMAIL_ADDRESS")

### Identifying header for outbound requests

Both SEC EDGAR and the OpenStreetMap geocoding service require a descriptive `User-Agent` header that identifies who is making the request. Failing to provide one results in a `403 Forbidden` response.

> **Required action.** Replace the placeholder string below with your actual name and Baruch email address before running the cell. Submissions that retain the placeholder will not run successfully against EDGAR.

In [ ]:
# REQUIRED: replace with your name and Baruch email address
USER_AGENT = f"CIS3120 Student {EMAIL_ADDRESS}"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

### A note on API etiquette

Public, no-cost APIs such as SEC EDGAR and OpenStreetMap Nominatim are operated as a service to researchers, journalists, and developers. They have no commercial revenue from individual users. To remain available, they impose two basic requirements:

1. **Identification.** Every request must carry a `User-Agent` header with a real name and contact email. This allows the service to contact a misbehaving client before resorting to an IP block.
2. **Rate limiting.** Each service publishes a maximum request rate. SEC EDGAR allows ten requests per second; OpenStreetMap Nominatim allows one request per second. Code that respects these limits with explicit `time.sleep()` calls is responsible practice and protects the resource for other users.

Violating these expectations typically results in a temporary IP block. For commercial workloads, paid services with higher rate limits and stronger guarantees are available, but for academic and prototype work the public services are sufficient.

## Stage 1 — Retrieve candidate 8-K filings from EDGAR

The first stage casts a wide net: it asks SEC EDGAR for every 8-K filing in the past 30 days that contains any one of a list of location-event keywords. Many of these candidates will turn out to be false positives — that is expected and is exactly why Stage 3 exists. The goal here is recall, not precision.

### Background: what EDGAR is and why this data exists

EDGAR (Electronic Data Gathering, Analysis, and Retrieval) is the SEC's public filing system. United States federal securities law requires publicly traded companies to disclose material events to the investing public on a continuous basis. Form 8-K is the specific filing used to report unscheduled material events between quarterly reports — including (relevant for this pipeline) the opening, closing, relocation, or expansion of significant facilities.

EDGAR has been online since 1994 and contains millions of filings. The full-text search endpoint at `efts.sec.gov/LATEST/search-index` is free, requires no API key, and indexes filings back to 2001 including their attached exhibits. Press releases announcing material events are typically attached to 8-K filings as Exhibit 99.1, which is the file the next stage will retrieve.

In [ ]:
import requests
import time
from datetime import date, timedelta

# Each phrase is queried independently, then results are merged.
# This avoids a parser limitation in the SEC's full-text search engine
# that prevents combining many quoted phrases with parenthetical OR groups.
SEARCH_PHRASES = [
    '"new store"',
    '"new facility"',
    '"new location"',
    '"distribution center"',
    '"fulfillment center"',
    '"grand opening"',
    '"store closure"',
    '"store closing"',
    '"facility closure"',
    '"opens new"',
]

# 30-day window ending today
WINDOW_DAYS = 30
WINDOW_END = date.today()
WINDOW_START = WINDOW_END - timedelta(days=WINDOW_DAYS)

EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"

print(f"Search window: {WINDOW_START} to {WINDOW_END} ({WINDOW_DAYS} days)")
print(f"Phrases to query: {len(SEARCH_PHRASES)}")

### Why we run one query per phrase instead of one combined query

A natural first attempt is to combine all phrases into a single query using boolean OR — for example, `"new store" OR "new facility" OR "distribution center"`. The SEC's documentation suggests this should work, but in practice the query parser is more restricted than the documentation describes. Combining many quoted phrases with parentheses produces zero hits, even for queries where each phrase individually returns hundreds of matches.

The robust workaround is to issue one search per phrase and merge the results. Each filing is uniquely identified by its accession number (the `_id` field on each search hit), so deduplication is straightforward: a Python dictionary keyed on accession number naturally collapses any filing that matches multiple phrases.

This pattern — discover a documented feature is broken in practice, then build a more resilient workaround — is a common situation in real analytical work. Documentation describes intent; observed behavior is the truth.

### Exercise S1 — Build the EDGAR query function

Implement `search_edgar_one_phrase(phrase, start_date, end_date, forms, max_pages)` so that it issues one or more paginated GET requests to the EDGAR full-text search endpoint and returns a tuple of `(hits_list, total_available_in_window)`.

**Required behavior:**

1. Build a `params` dictionary with the keys `q`, `dateRange`, `startdt`, `enddt`, `forms`, and `from`. The first five take the obvious values; `from` is the pagination offset (`page * 100`).
2. Issue the request with `requests.get(EDGAR_SEARCH_URL, params=params, headers=REQUEST_HEADERS, timeout=30)`. Call `response.raise_for_status()` to fail loudly on HTTP errors.
3. Parse the JSON response. Hits are at `data["hits"]["hits"]`; total available is at `data["hits"]["total"]["value"]`.
4. Accumulate hits across pages. Stop early when `(page + 1) * 100 >= total`.
5. Pause `0.15` seconds between paginated requests to respect SEC's ten-requests-per-second limit.
6. Return `(all_hits, total)`.

The function signature, docstring, and a `raise NotImplementedError` placeholder are provided. Replace the placeholder with your implementation.

### Stepwise construction of the EDGAR query function

Rather than writing the entire `search_edgar_one_phrase` function in one attempt, the next several cells build it one component at a time. Each step adds a single piece of behavior, runs it in isolation so its output is visible, and then proceeds to the next piece. The final cell in the sequence assembles every piece into the complete function.

The six conceptual pieces the function must perform are:

1. Build a parameter dictionary that the EDGAR endpoint will accept.
2. Issue one HTTP GET request and check it succeeded.
3. Parse the JSON response and locate the hits list and total count.
4. Repeat steps 1 through 3 across multiple pages (pagination).
5. Stop paginating early once every available hit has been retrieved.
6. Pause briefly between paginated requests so as not to exceed the SEC's ten-requests-per-second limit.

Each subsection below addresses one of these pieces in order.

#### Step 1 — Build the parameter dictionary

The EDGAR full-text search endpoint expects parameters supplied as URL query-string arguments. The `requests` library accepts these as a Python dictionary passed via the `params` keyword argument; `requests` then serializes the dictionary into the URL automatically.

Six parameter keys are required:

- `q` — the search phrase, in quotation marks.
- `dateRange` — fixed string `"custom"` to indicate that explicit dates follow.
- `startdt` — start date as an ISO-format string (`YYYY-MM-DD`).
- `enddt` — end date as an ISO-format string.
- `forms` — the form type, here `"8-K"`.
- `from` — the pagination offset, equal to the page index multiplied by 100 (the page size). For the first page this is 0, for the second page this is 100, and so on.

The cell below constructs this dictionary for page 0 of the phrase `"new store"` over the configured 30-day window. Print the dictionary to verify the values are what you expect before moving on.

In [ ]:
page = 0
phrase_demo = '"new store"'

params_demo = {
    "q": phrase_demo,
    "dateRange": "custom",
    "startdt": WINDOW_START.isoformat(),
    "enddt": WINDOW_END.isoformat(),
    "forms": "8-K",
    "from": page * 100,
}

params_demo

#### Step 2 — Issue one HTTP GET request

With the parameter dictionary built, the next step is to issue the request itself. The `requests.get` call takes four arguments here: the URL, the params dictionary just constructed, the `REQUEST_HEADERS` dictionary containing the User-Agent string set up earlier, and a `timeout` value that prevents the request from hanging indefinitely if the network stalls.

Immediately after the call, `response.raise_for_status()` is invoked. This method does nothing on success but raises an exception on any HTTP error response (4xx or 5xx). Calling it explicitly prevents silent failures: without this line, a transient SEC error would be ignored and the function would return an empty hits list as if the search itself had found nothing.

After the call, `response.status_code` should print as `200`.

In [ ]:
REQUEST_HEADERS

In [ ]:
response_demo = requests.get(
    EDGAR_SEARCH_URL,
    params=params_demo,
    headers=REQUEST_HEADERS,
    timeout=30,
)

try:
    response_demo.raise_for_status()
except requests.HTTPError as exc:
    print("Request failed.")
    print("  Status code:    ", response_demo.status_code)
    print("  URL:            ", response_demo.url)
    print("  Sent User-Agent:", repr(response_demo.request.headers.get("User-Agent")))
    print(
        "  If the User-Agent above looks malformed (empty, contains the "
        "placeholder, or contains hidden characters), correct the cell that "
        "defines USER_AGENT and rerun. Otherwise this is most likely a "
        "transient SEC server error; wait briefly and retry."
    )
    raise


print("HTTP status code:", response_demo.status_code)

#### Step 3 — Parse the JSON response and locate the hits

EDGAR returns its results as JSON. The `response.json()` method deserializes the response body into a nested Python dictionary.

The structure of that dictionary is worth examining briefly because the function will need to navigate it. The two pieces of information needed are:

- `data["hits"]["hits"]` — a list of dictionaries, one per matching filing. Each entry contains the filing's accession ID, file date, company name, and other metadata.
- `data["hits"]["total"]["value"]` — an integer giving the total number of matches available for this query, regardless of pagination. This number is used in Step 5 to decide when to stop paginating.

The cell below extracts both values, then prints how many hits this single page returned and how many are available in total.

In [ ]:
data_demo = response_demo.json()

hits_demo = data_demo.get("hits", {}).get("hits", [])
total_demo = data_demo.get("hits", {}).get("total", {}).get("value", 0)

print(f"Hits returned on this page: {len(hits_demo)}")
print(f"Total hits available in window: {total_demo}")

Inspect the first hit to see what fields are available. The fields used later in the pipeline are `_id` (the accession ID and exhibit filename), and the nested `_source` dictionary which contains `display_names`, `tickers`, `file_date`, and `ciks`.

In [ ]:
if hits_demo:
    first_hit = hits_demo[0]
    print("_id:        ", first_hit["_id"])
    print("display_names:", first_hit["_source"].get("display_names"))
    print("file_date:    ", first_hit["_source"].get("file_date"))
    print("tickers:      ", first_hit["_source"].get("tickers"))

#### Step 4 — Wrap Steps 1 through 3 in a pagination loop

EDGAR returns at most 100 hits per request. To retrieve more than 100, additional requests must be issued with the `from` parameter incremented by 100 each time. This is called **pagination**, and a `for` loop is the natural Python construct for it.

The loop variable `page` ranges from 0 up to (but not including) `max_pages`. At each iteration the only thing that changes in the parameter dictionary is the value of `from`, which is set to `page * 100`. Hits from each page are accumulated into a single list called `all_hits` using `list.extend()`.

The cell below performs two pages of pagination for the same phrase as the earlier steps, accumulating the results.

In [ ]:
max_pages_demo = 2
all_hits_demo = []
total_demo = 0

for page in range(max_pages_demo):
    params = {
        "q": phrase_demo,
        "dateRange": "custom",
        "startdt": WINDOW_START.isoformat(),
        "enddt": WINDOW_END.isoformat(),
        "forms": "8-K",
        "from": page * 100,
    }
    response = requests.get(
        EDGAR_SEARCH_URL, params=params,
        headers=REQUEST_HEADERS, timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    page_hits = data.get("hits", {}).get("hits", [])
    all_hits_demo.extend(page_hits)
    total_demo = data.get("hits", {}).get("total", {}).get("value", 0)
    print(f"Page {page}: retrieved {len(page_hits)} hits (total available: {total_demo})")

print(f"\nTotal hits accumulated across {max_pages_demo} pages: {len(all_hits_demo)}")

#### Step 5 — Stop paginating once every hit has been retrieved

If a query returns only 47 total hits, requesting page 1 (offsets 100 through 199) is wasted effort because there is nothing left to retrieve. An early-stop check after each page prevents this waste.

The condition is: after processing page `p`, the number of records retrieved so far is `(p + 1) * 100`. If this equals or exceeds the total available, every hit has been collected and the loop should `break`.

The comparison is `>=` rather than `>`. The `>=` version correctly handles the edge case where the last page is exactly full — for example, if `total` equals 200 and the second page (`p = 1`) returns the final 100 hits, then `(1 + 1) * 100` equals `200` which equals `total`, and the loop should stop. A `>` comparison would issue an unnecessary third request that returns nothing.

The cell below adds this check to the pagination loop.

In [ ]:
max_pages_demo = 5  # raised so the early-stop condition can fire
all_hits_demo = []
total_demo = 0

for page in range(max_pages_demo):
    params = {
        "q": phrase_demo,
        "dateRange": "custom",
        "startdt": WINDOW_START.isoformat(),
        "enddt": WINDOW_END.isoformat(),
        "forms": "8-K",
        "from": page * 100,
    }
    response = requests.get(
        EDGAR_SEARCH_URL, params=params,
        headers=REQUEST_HEADERS, timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    page_hits = data.get("hits", {}).get("hits", [])
    all_hits_demo.extend(page_hits)
    total_demo = data.get("hits", {}).get("total", {}).get("value", 0)
    print(f"Page {page}: retrieved {len(page_hits)} hits")

    if (page + 1) * 100 >= total_demo:
        print(f"  Early stop: retrieved {(page + 1) * 100} >= total {total_demo}")
        break

print(f"\nFinal hits retrieved: {len(all_hits_demo)} of {total_demo} available")

#### Step 6 — Pause briefly between paginated requests

The SEC publishes a courtesy limit of ten requests per second on its free public endpoints. Sleeping `0.15` seconds between paginated requests keeps the function comfortably under that limit (one request every 0.15 seconds is approximately 6.7 requests per second). This is good API citizenship and prevents the IP address from being temporarily blocked during heavy use.

The sleep is placed at the bottom of the loop body so that it runs after each request but is skipped when the early-stop `break` fires.

#### Step 7 — Assemble the complete function

All six pieces above are now combined into a single function with a `phrase` parameter, a `start_date` parameter, an `end_date` parameter, and the optional `forms` and `max_pages` parameters that have default values. The function returns a tuple of two values: the accumulated hits list and the total available count. The total is returned alongside the hits so that the caller can report "5 hits retrieved out of 47 available" if it wants to.

Compare this assembled version to the six steps above; every line in the body corresponds to one of those steps.

In [ ]:
def search_edgar_one_phrase(phrase, start_date, end_date, forms="8-K", max_pages=2):
    """Run a single-phrase EDGAR full-text search.

    Returns a tuple of (hits_list, total_available_in_window). The endpoint
    paginates with up to 100 hits per page; we cap at max_pages pages.
    """
    all_hits = []
    total = 0
    for page in range(max_pages):
        params = {
            "q": phrase,
            "dateRange": "custom",
            "startdt": start_date.isoformat(),
            "enddt": end_date.isoformat(),
            "forms": forms,
            "from": page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(0.15)  # respect SEC's 10 req/sec rate limit
    return all_hits, total

Verify the assembled function works by calling it once with the same arguments used in the stepwise build-up. The result should match the Step 5 output exactly.

In [ ]:
verify_hits, verify_total = search_edgar_one_phrase(
    '"new store"', WINDOW_START, WINDOW_END
)
print(f"Function returned {len(verify_hits)} hits out of {verify_total} available.")

> **💬 DISCUSSION PROMPT**
>
> We discovered that combining many phrases with OR returns zero hits, even though the SEC documentation says it should work. What does this teach us about how to validate that an API actually does what its documentation describes?

### Applying the function to every search phrase

With `search_edgar_one_phrase` now defined, the cell below calls it once per phrase in `SEARCH_PHRASES` and accumulates results into a single dictionary keyed by accession ID. Using the accession ID as the dictionary key automatically deduplicates filings that match more than one search phrase.

In [ ]:
# Run one query per phrase, deduplicate results by accession ID
all_hits_by_id = {}
print(f"{'Phrase':<25} {'Total in window':>18}")
print("-" * 45)
for phrase in SEARCH_PHRASES:
    hits, total = search_edgar_one_phrase(phrase, WINDOW_START, WINDOW_END)
    print(f"{phrase:<25} {total:>18}")
    for hit in hits:
        all_hits_by_id[hit["_id"]] = hit
    time.sleep(0.15)

candidates = list(all_hits_by_id.values())
total_found = len(candidates)
print(f"\nUnique candidates after deduplication: {total_found}")

### Inspect the candidate set

Before sending anything to the LLM, look at the raw retrieval. Each EDGAR hit contains the company name, filing date, accession number, and the exhibit filename containing the press release text. A quick visual scan often reveals patterns — for example, a single company filing many 8-Ks in a short window, or filings from companies whose names suggest they probably do not operate physical facilities.

In [ ]:
# Show the first 10 candidates so we can visually inspect what came back
print(f"{'Date':<12} {'Company':<45} {'Exhibit'}")
print("-" * 100)
for hit in candidates[:10]:
    src = hit["_source"]
    company = (src.get("display_names") or ["(unknown)"])[0]
    if len(company) > 43:
        company = company[:42] + "…"
    file_date = src.get("file_date", "")
    exhibit = hit["_id"].split(":")[-1] if ":" in hit["_id"] else hit["_id"]
    print(f"{file_date:<12} {company:<45} {exhibit}")

## Stage 2 — Fetch the press release text from each filing

The search results from Stage 1 contain metadata about each filing but not the press release text itself. This stage downloads each exhibit file and converts it to clean plain text suitable for the LLM in Stage 3.

Stage 2 is provided complete. You do not need to modify any code in this stage.

### Background: what an "exhibit" is and how we locate the file

An 8-K filing is structured: a short cover document declares which "items" the filing reports (Item 8.01 for "Other Events," Item 2.05 for "Costs Associated with Exit Activities," and so on), and one or more exhibits provide the supporting material. For announcements of facility openings or closings, the substantive content almost always lives in **Exhibit 99.1**, which is the press release the company's communications team issued.

EDGAR stores every filing under a stable URL pattern:

```
https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_dashes}/{filename}
```

The CIK (Central Index Key) is the SEC's unique identifier for each company, and the accession number is the unique identifier for each filing. Both are present in the search hit returned by Stage 1, which is what `build_exhibit_url()` below assembles into a complete URL.

In [ ]:
from bs4 import BeautifulSoup


def build_exhibit_url(hit):
    """Construct the canonical EDGAR archive URL for the exhibit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_dashes}/{filename}"


def fetch_exhibit_text(hit, max_chars=8000):
    """Download the exhibit, strip HTML to plain text, and truncate.

    Returns (clean_text, exhibit_url).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()

    # Convert HTML markup to plain text
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)

    # Truncate long releases to control LLM input cost in Stage 3
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"

    return text, url


# Smoke test: fetch the first candidate and inspect the first 500 characters
sample_text, sample_url = fetch_exhibit_text(candidates[0])
print(f"URL: {sample_url}")
print(f"Length: {len(sample_text)} characters")
print(f"\nFirst 500 characters:\n{sample_text[:500]}")

### Why HTML stripping and truncation are needed

EDGAR serves exhibits in HTML so they render correctly in a browser. For an LLM that processes plain text, HTML markup is noise: it inflates the token count without adding meaning, and complex layouts can confuse the model about which text is the substantive content versus boilerplate (footers, page numbers, exhibit indexes).

The `BeautifulSoup` library walks the parsed HTML tree and extracts only the visible text content, discarding tags and inline styling. The result is the press release as a continuous block of plain text.

The 8,000-character truncation is a deliberate cost-control measure. Most press releases are between 1,500 and 4,000 characters and pass through unchanged; the truncation only affects unusually long filings such as multi-event announcements or attached financial tables. Eight thousand characters maps to roughly 2,000 tokens — enough context for the LLM to determine event type and location, while keeping per-filing input cost predictable.

### Fetch all candidate exhibits

The loop below fetches every candidate. With approximately 100 candidates and a 0.15-second pause per request (well under the SEC's published limit of ten requests per second), the loop completes in roughly thirty seconds. Failures are logged but do not halt the pipeline.

In [ ]:
fetched_filings = []
fetch_failures = 0

for i, hit in enumerate(candidates):
    try:
        text, url = fetch_exhibit_text(hit)
        src = hit["_source"]
        fetched_filings.append({
            "company": (src.get("display_names") or ["(unknown)"])[0],
            "ticker": (src.get("tickers") or [None])[0],
            "file_date": src.get("file_date"),
            "accession": hit["_id"].split(":")[0],
            "url": url,
            "text": text,
        })
    except Exception as exc:
        fetch_failures += 1
        print(f"  [warn] Failed to fetch {hit['_id']}: {exc}")

    time.sleep(0.15)  # respect SEC's 10 req/sec rate limit

    if (i + 1) % 25 == 0:
        print(f"Fetched {i + 1} of {len(candidates)}…")

print(f"\nFetched {len(fetched_filings)} filings successfully.")
print(f"Failures: {fetch_failures}")

## Stage 3 — Classify and extract with the Anthropic API

For each fetched filing, the pipeline asks Claude two questions in a single call: (1) does this filing genuinely describe a location-related event, and (2) if so, what are the structured details (event type, city, state, brief summary). Filings that fail the first check are discarded; the rest are passed to Stage 4 for geocoding.

### Background: asking the model for structured output

Module 14 demonstrated how to send free-form prompts to Claude and receive free-form text responses. For analytical pipelines this is rarely useful — downstream code needs values it can address by name, not paragraphs of prose to parse.

The technique here is **structured output via prompt engineering**: the system prompt explicitly instructs the model to return only a JSON object with a specified schema. The downstream code parses that JSON with `json.loads()` and accesses fields directly (`extraction["event_type"]`, `extraction["city"]`, and so on).

Two practical considerations when designing this kind of prompt:

1. **The schema must be unambiguous.** Specify the exact field names, the type of each field (string, boolean, null), and the allowed values for any enumerated field. Vague specifications produce inconsistent output.
2. **Defensive parsing matters.** Even with explicit instructions, models occasionally wrap JSON in markdown fences, add a one-line preamble, or hallucinate an extra field. The `re.sub` line in the function below strips markdown fences if they appear; a `try/except` around `json.loads()` catches the rare malformed response without halting the loop.

### Why Haiku 4.5 is the right model for this task

Module 14 used Claude Opus 4.7 for general-purpose generation. Opus is the most capable model in the Claude family, but capability is not the only consideration — for high-volume, well-bounded tasks, a smaller model is the better engineering choice.

This pipeline performs a binary classification (is this a location event, yes or no) followed by structured extraction of a handful of fields from short text. That workload is precisely what **Claude Haiku 4.5** is designed for. At one dollar per million input tokens (compared to five dollars for Opus), it reduces the per-run cost by approximately eighty percent while producing equivalent results on this kind of well-specified task. Using Opus here would be analogous to renting a freight truck to deliver a single envelope — the smaller vehicle is the more responsible choice when the workload does not justify the larger one.

The general principle: match the model to the task. Reserve the most capable models for the parts of a pipeline that genuinely require deep reasoning, and use smaller, cheaper models for routine extraction and classification.

### Exercise S3 — Write the extraction system prompt

This is the highest-leverage exercise in the notebook. The system prompt is the single artifact that determines whether the LLM stage succeeds or fails. A vague prompt produces inconsistent JSON; a strict prompt produces clean, parseable records that the rest of the pipeline can rely on.

Your task is to write `EXTRACTION_SYSTEM_PROMPT` as a Python string. The prompt must instruct Claude to:

1. Adopt the role of an analyst reviewing 8-K filing exhibits to identify location-related corporate events (openings, closings, relocations, expansions of physical facilities such as stores, warehouses, distribution centers, offices, plants).
2. Return **only** a JSON object with these exact fields:
   - `is_location_event` — boolean. True only when the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, or generic corporate updates.
   - `event_type` — one of `"opening"`, `"closing"`, `"relocation"`, `"expansion"`, `"other"`, or null.
   - `city` — string with the city name, or null.
   - `state` — two-letter US state code (e.g. `"NY"`, `"CA"`), or null if not US-based or not specified.
   - `summary` — one sentence under 25 words describing the event in plain language, or null.
3. Be strict: locations mentioned only in passing (such as the headquarters address in boilerplate) should yield `is_location_event: false`.
4. Return only the JSON object — no preamble, no markdown fences, no explanation.

The function shell `extract_with_claude()` and the user-message format are provided. Write the system prompt only.

### Stepwise construction of the extraction system prompt

A system prompt is a single Python string. Writing it as one long block from scratch makes it hard to reason about which clauses do which work. The walkthrough below decomposes the prompt into four distinct parts and writes each part as its own short string. The final step concatenates the four parts into the complete prompt.

The four parts of an extraction prompt are:

1. **Role statement** — tells the model what perspective to adopt and what task to perform.
2. **Schema specification** — enumerates each field name and its expected type. This is the most important section because it determines whether the downstream JSON parser will succeed.
3. **Strictness clauses** — explicit examples of inputs that should yield `is_location_event: false`. Vague negative instructions are far less reliable than enumerated negative examples.
4. **Output discipline clause** — a short, direct instruction that the response must contain only the JSON object and nothing else (no preamble, no markdown fences, no explanation).

Each part is built and inspected in its own cell below.

#### Step 1 — Role statement

The opening sentence establishes the role. "You are an analyst reviewing 8-K filing exhibits" anchors the model in a specific domain and gives it a frame for interpreting the input. The next clause names the four event types of interest and gives examples of the kinds of physical facilities they apply to. Naming examples is more effective than abstract definition: the phrase "stores, warehouses, distribution centers, offices, plants" gives the model concrete categories to match against the filing text.

Notice that the role statement does not yet say anything about JSON or output format. Each part of the prompt has one job; the role statement establishes the task only.

In [ ]:
role_part = (
    "You are an analyst reviewing 8-K filing exhibits to identify "
    "announcements of location-related corporate events: openings, "
    "closings, relocations, or expansions of physical facilities "
    "(stores, warehouses, distribution centers, offices, plants)."
)

print(role_part)

#### Step 2 — Schema specification

The schema lists every field the JSON response must contain, the type each field must have, and (where applicable) the allowed values. Two details matter most here:

- The `event_type` field is constrained to one of five string values: `"opening"`, `"closing"`, `"relocation"`, `"expansion"`, or `"other"`. Without this enumeration the model will invent its own category names (`"new_opening"`, `"facility_closure"`) which break the downstream `EVENT_COLORS` dictionary that maps event types to marker colors.
- The `state` field is constrained to a two-letter US state code (`"NY"`, `"CA"`). Without this constraint the model will sometimes return `"California"` and sometimes `"CA"`, which would force the geocoding stage to handle both forms.

Every field also lists `null` as an acceptable value for the cases where the information is not present. This prevents the model from fabricating a value when the filing genuinely does not contain one.

In [ ]:
schema_part = (
    "\n\nReturn ONLY a JSON object with these exact fields:\n"
    "- is_location_event: boolean. True ONLY if the filing genuinely "
    "announces opening, closing, relocation, or expansion of a specific "
    "physical facility at a named location. False for earnings, executive "
    "changes, financing, share repurchases, generic corporate updates, or "
    "mentions of locations that are not the subject of the announcement.\n"
    "- event_type: one of \"opening\", \"closing\", \"relocation\", "
    "\"expansion\", \"other\", or null\n"
    "- city: string with the city name, or null if no specific city is named\n"
    "- state: two-letter US state code (e.g., \"NY\", \"CA\"), or null "
    "if not US-based or not specified\n"
    "- summary: one sentence (under 25 words) describing the event in plain "
    "language, or null"
)

print(schema_part)

#### Step 3 — Strictness clause

The strictness clause addresses a specific failure mode: filings that mention a location only in passing. Many 8-K filings include the company's headquarters address as part of standard boilerplate at the top or bottom of the document. Without an explicit instruction to ignore such mentions, the model will sometimes treat them as location events and produce a false positive — for example, classifying a routine earnings filing as a New York opening because the company's headquarters appears in the document.

The clause is written as a single sentence with one concrete example ("headquarters address in the boilerplate"). One concrete example carries more weight than a paragraph of abstract guidance.

In [ ]:
strictness_part = (
    "\n\nBe strict. If the filing mentions a location only in passing "
    "(e.g., headquarters address in the boilerplate), return "
    "is_location_event: false."
)

print(strictness_part)

#### Step 4 — Output discipline clause

The output discipline clause is a single sentence at the end of the prompt that names the three things the response must not contain: a preamble, markdown fences, or an explanation. Each of these is a specific behavior that LLMs will sometimes perform by default and that would each break the JSON parser:

- A preamble such as "Here is the structured extraction:" prepended to the JSON would make `json.loads` fail because the input would not start with `{`.
- Markdown code fences (` ```json ... ``` `) wrap the JSON in a code block. The downstream code includes a defensive regex to strip these, but it is more reliable to instruct the model not to add them at all.
- An explanation appended after the JSON would make `json.loads` fail because of trailing content.

Naming all three failure modes explicitly is more effective than a generic instruction such as "return only JSON."

In [ ]:
output_part = (
    " Return only the JSON object with no preamble, no markdown fences, "
    "no explanation."
)

print(output_part)

#### Step 5 — Concatenate the four parts into the final prompt

With each part written and verified independently, the full prompt is produced by concatenating the four strings in order. Print the result to confirm it reads as a single coherent instruction rather than four disconnected fragments.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = role_part + schema_part + strictness_part + output_part

print(EXTRACTION_SYSTEM_PROMPT)

#### Step 6 — Wire the prompt into the API call

The `extract_with_claude` function below takes one filing dictionary, constructs the user message by concatenating the company name, file date, and exhibit text, and sends one request to the Anthropic API. The system prompt argument is the `EXTRACTION_SYSTEM_PROMPT` string constructed above; the model id is `claude-haiku-4-5-20251001`, the model selected for high-volume bounded extraction tasks.

After receiving the response, two layers of defensive parsing are applied. The first uses a regex to strip markdown fences if the model added any despite the prompt instruction. The second wraps the `json.loads` call in a try-except that catches malformed JSON and returns a sentinel record rather than crashing the entire pipeline. This means a single bad extraction does not halt processing of the remaining filings.

In [ ]:
import anthropic
import json
import re

client = anthropic.Anthropic()
MODEL_ID = "claude-haiku-4-5-20251001"


def extract_with_claude(filing):
    """Send filing text to Claude and parse the JSON response."""
    user_message = f"Filing from {filing['company']} dated {filing['file_date']}:\n\n{filing['text']}"

    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=400,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )

    raw = response.content[0].text.strip()

    # Defensive parsing: strip markdown fences if the model added them
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"is_location_event": False, "_parse_error": raw[:200]}

> **💬 DISCUSSION PROMPT**
>
> We chose Haiku 4.5 instead of Opus 4.7 for this stage. Walk through the reasoning. What characteristics of this task made Haiku the appropriate choice?

In [ ]:
# Smoke test on the first filing
sample_extraction = extract_with_claude(fetched_filings[0])
print(json.dumps(sample_extraction, indent=2))

### The recall vs precision tradeoff that this pipeline manages

This is a useful moment to introduce a concept that recurs throughout analytics. Information retrieval systems are typically evaluated on two complementary measures:

- **Recall** is the fraction of relevant items the system actually retrieves. If twenty filings in the past 30 days truly describe new store openings and the system finds eighteen of them, recall is 90 percent.
- **Precision** is the fraction of retrieved items that are actually relevant. If the system returns 100 candidates and only 18 are real openings, precision is 18 percent.

These two measures usually trade off against each other. Tightening the keyword list raises precision but loses some real events (lower recall). Loosening it raises recall but admits more false positives (lower precision).

The pipeline you are running uses a deliberate division of labor:

1. **Stage 1 optimizes for recall.** Broad keyword search catches almost every relevant filing, accepting that many irrelevant ones come along.
2. **Stage 3 supplies the precision.** The LLM reads each candidate and discards the ones that are not genuine location events.

When you run the next cell, watch the ratio in the classification summary. A typical result is approximately 12 to 18 percent of candidates classified as genuine events. That percentage is not a problem with the pipeline — it is the precision the LLM contributes, and the demonstration that the LLM stage is doing real work.

### Run extraction on every filing

The loop below processes one filing per API call. With approximately 100 candidates at Haiku 4.5 pricing, the full run costs less than fifty cents. The `max_filings` cap remains as a safety bound; raise or lower it to control how many filings are processed.

In [ ]:
max_filings = 250  # Haiku makes the full candidate set affordable

events = []
classification_counts = {"location_event": 0, "not_location_event": 0, "error": 0}

for i, filing in enumerate(fetched_filings[:max_filings]):
    try:
        extraction = extract_with_claude(filing)
        if extraction.get("is_location_event"):
            classification_counts["location_event"] += 1
            events.append({**filing, **extraction})
        else:
            classification_counts["not_location_event"] += 1
    except Exception as exc:
        classification_counts["error"] += 1
        print(f"  [warn] Extraction failed for {filing['company']}: {exc}")

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1} of {min(max_filings, len(fetched_filings))}…")

print(f"\nClassification summary:")
for label, count in classification_counts.items():
    print(f"  {label}: {count}")
print(f"\nLocation events to plot: {len(events)}")

### Inspect the extracted events

Before geocoding, review a sample of what the LLM identified. This is the moment to spot-check that the classifications make sense — a quick scan can reveal whether the model is consistently catching real events and consistently rejecting false positives, or whether the prompt needs adjustment.

In [ ]:
for event in events[:10]:
    print(f"{event['file_date']} | {event['company']}")
    print(f"  Type:    {event['event_type']}")
    print(f"  Where:   {event.get('city')}, {event.get('state')}")
    print(f"  Summary: {event['summary']}")
    print()

## Stage 4 — Geocode the locations

The structured records from Stage 3 contain city and state names but not coordinates. Folium plots markers from latitude and longitude, so each location must first be converted to a coordinate pair. This step is called **geocoding**.

Stage 4 is provided complete. You do not need to modify any code in this stage.

### Background: what geocoding is and which service we use

Geocoding is the process of converting a human-readable place description ("Memphis, TN") into geographic coordinates (latitude 35.149, longitude -90.049). The reverse operation — coordinates to address — is called reverse geocoding.

Several geocoding services are available. Google Maps and Mapbox offer high-quality geocoding with permissive rate limits, but both require API keys and become expensive at volume. The free alternative, used here, is **OpenStreetMap Nominatim**, which is operated by the OpenStreetMap Foundation as a public service. Nominatim is free and requires no API key, but it enforces a strict limit of one request per second and requires a descriptive `User-Agent` header.

Two failure modes are common with city-level geocoding and worth being aware of:

1. **Ambiguous city names.** Many cities share names with other places — "Springfield" alone is unresolvable because it exists in many states. Including the state in the query, as the function below does, almost always resolves this.
2. **Non-US locations.** The pipeline appends `, USA` to every query for consistency. Filings about international facilities will fail to geocode and are dropped at this stage. For an internationalized version of the lesson, the city-state-country logic would need to be more flexible.

In [ ]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"


def geocode_location(city, state):
    """Convert a city/state pair to (latitude, longitude). Returns None on failure."""
    if not city:
        return None

    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1}

    try:
        response = requests.get(
            NOMINATIM_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        results = response.json()
        if results:
            return float(results[0]["lat"]), float(results[0]["lon"])
    except Exception as exc:
        print(f"  [warn] Geocoding failed for '{query}': {exc}")

    return None


# Geocode every event from Stage 3
geocoded_events = []
for event in events:
    coords = geocode_location(event.get("city"), event.get("state"))
    if coords:
        event["latitude"], event["longitude"] = coords
        geocoded_events.append(event)
    time.sleep(1.1)  # Nominatim requires <= 1 request per second

print(f"Successfully geocoded: {len(geocoded_events)} of {len(events)} events")

## Stage 5 — Render the folium map

The final stage converts the geocoded events into an interactive map. Each event becomes a colored marker; clicking the marker reveals a popup with the company name, filing date, summary, and a link to the underlying SEC filing.

### Background: visualization design choices

A small number of decisions in the code below collectively determine how readable the final map is. Each is worth understanding rather than treating as boilerplate.

**Tile choice: CartoDB Positron.** Folium can use any of several base map providers. CartoDB Positron is a deliberately muted gray-and-white style designed specifically for data visualization — its low visual contrast lets the data markers stand out instead of competing with detailed geography. The default OpenStreetMap tiles, by contrast, include heavy color and labeling that make small markers harder to see.

**Color encoding by event type.** The `EVENT_COLORS` dictionary assigns a distinct color to each event type, providing a visual taxonomy that the eye can decode at a glance: green for openings (positive), red for closings (negative), orange for relocations (neutral movement), blue for expansions (positive growth), gray for ambiguous "other" events. This mapping follows widely understood color conventions; choosing colors that match audience intuition reduces the cognitive load of reading the map.

**Popups rather than always-visible labels.** With twenty to thirty markers across the United States, always-visible labels would overlap and clutter the map. Popups (revealed only on click) keep the default view clean while making the underlying detail available on demand. This is a common pattern: the high-level view shows distribution and density; interaction reveals individual records.

**Centering on the data.** Setting the initial map view to the geographic mean of the events ensures most markers are visible without manual panning. For a US-focused dataset this typically lands near the center of the continental United States.

### Exercise S5 — Configure the folium marker loop

The cell below provides the fixed scaffolding: the `EVENT_COLORS` dictionary, the centering calculation, and the empty `folium.Map(...)` object. Your task is to write the loop that adds one marker per event.

**Required behavior for each marker:**

1. Look up the marker color from `EVENT_COLORS` using `event.get("event_type")`. Default to `"gray"` for unknown event types.
2. Compose a popup HTML string that displays, in order: the company name (bold), the filing date and event type (italic, on one line), the summary, and a link to the SEC filing (`event['url']`) that opens in a new tab. Wrap the whole popup in a fixed-width `<div>` of about 260 pixels.
3. Construct a `folium.Marker(...)` at `[event['latitude'], event['longitude']]` with the popup, a tooltip showing the company name, and `folium.Icon(color=color, icon="info-sign")`.
4. Call `.add_to(m)` so the marker attaches to the map.

Build the loop where indicated. The final `m` on its own line at the bottom of the cell renders the interactive map inline.

### Stepwise construction of the folium marker loop

The marker loop is the smallest of the three exercises but combines three distinct skills: dictionary lookup with a default, Python string composition for inline HTML, and the folium API for adding layers to a map. The walkthrough below treats each of these as its own step.

Before constructing the loop, the next cell sets up the fixed scaffolding used throughout: the `EVENT_COLORS` dictionary, the centering calculation, and the empty `folium.Map(...)` object. This is the same scaffolding shown in the exercise prompt above.

In [ ]:
import folium

EVENT_COLORS = {
    "opening": "green",
    "closing": "red",
    "relocation": "orange",
    "expansion": "blue",
    "other": "gray",
}

# Center the map on the geographic mean of the events
if geocoded_events:
    center_lat = sum(e["latitude"] for e in geocoded_events) / len(geocoded_events)
    center_lon = sum(e["longitude"] for e in geocoded_events) / len(geocoded_events)
else:
    center_lat, center_lon = 39.5, -98.35  # geographic center of contiguous US

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=4,
    tiles="CartoDB positron",
)

#### Step 1 — Look up the marker color

Each event has an `event_type` field whose value is one of `"opening"`, `"closing"`, `"relocation"`, `"expansion"`, or `"other"`. The `EVENT_COLORS` dictionary maps each of these to a folium-supported color name. The lookup uses `dict.get(key, default)` rather than direct subscript access for two reasons:

- If the event type is missing or null, `event['event_type']` would raise a `KeyError` and crash the loop.
- The fallback value `"gray"` keeps the visual contract consistent: every event gets a marker, even those whose category cannot be determined.

The cell below demonstrates the lookup for the first geocoded event.

In [ ]:
if geocoded_events:
    sample_event = geocoded_events[0]
    sample_color = EVENT_COLORS.get(sample_event.get("event_type"), "gray")
    print(f"Event type: {sample_event.get('event_type')!r}")
    print(f"Marker color: {sample_color!r}")

#### Step 2 — Compose the popup HTML

When a user clicks a marker on the map, folium displays a popup window containing whatever HTML was supplied at marker creation time. Composing this HTML using a single Python f-string is the most direct approach.

The popup contains four pieces of information arranged in vertical order:

1. The company name, in bold (`<b>...</b>`).
2. The file date and event type, in italics on a single line (`<i>...</i>`).
3. The summary sentence in normal text.
4. A link to the underlying SEC filing, with `target="_blank"` so it opens in a new browser tab and does not navigate the user away from the map.

All four pieces are wrapped in a `<div>` with an explicit width of 260 pixels. Without this width, folium will let the popup expand to fit the longest line — a long company name like "International Business Machines Corporation" can stretch a popup wide enough to look broken.

Inline `style` attributes are used rather than CSS classes because folium serializes each popup independently and does not share a stylesheet across markers.

In [ ]:
if geocoded_events:
    sample_event = geocoded_events[0]
    sample_popup_html = (
        f"<div style='width:260px'>"
        f"<b>{sample_event['company']}</b><br>"
        f"<i>{sample_event['file_date']} — {sample_event['event_type']}</i><br>"
        f"{sample_event['summary']}<br>"
        f"<a href='{sample_event['url']}' target='_blank'>View SEC filing</a>"
        f"</div>"
    )
    print(sample_popup_html)

#### Step 3 — Construct one marker

A `folium.Marker` is constructed with four arguments:

- `location` — a two-element list of `[latitude, longitude]`. Both are floats produced by Stage 4.
- `popup` — a `folium.Popup` object wrapping the HTML string from Step 2. The `max_width=300` keyword argument constrains how wide folium will let the popup grow; the inner `<div>` width controls the default presentation, while `max_width` provides a hard upper bound.
- `tooltip` — the company name, displayed when the user hovers over the marker without clicking. This gives quick context without requiring a click.
- `icon` — a `folium.Icon` configured with the color from Step 1 and the visual glyph `"info-sign"`.

The cell below constructs but does not yet attach the marker. The next step explains the attachment.

In [ ]:
if geocoded_events:
    sample_marker = folium.Marker(
        location=[sample_event["latitude"], sample_event["longitude"]],
        popup=folium.Popup(sample_popup_html, max_width=300),
        tooltip=sample_event["company"],
        icon=folium.Icon(color=sample_color, icon="info-sign"),
    )
    print(type(sample_marker).__name__, "constructed.")

#### Step 4 — Add the marker to the map

Constructing a `folium.Marker` object does not, by itself, place the marker on any map. The marker must be attached to the target map object using the `add_to(m)` method. Forgetting this call is the most common error in this exercise: the loop runs without raising an exception, but the rendered map has no markers.

By convention, `add_to(m)` is chained directly onto the marker constructor in a single expression. This keeps the construct-and-attach operation visually together and makes it harder to forget the attachment.

In [ ]:
if geocoded_events:
    folium.Marker(
        location=[sample_event["latitude"], sample_event["longitude"]],
        popup=folium.Popup(sample_popup_html, max_width=300),
        tooltip=sample_event["company"],
        icon=folium.Icon(color=sample_color, icon="info-sign"),
    ).add_to(m)

    print("Sample marker attached to map.")

#### Step 5 — Wrap Steps 1 through 4 in a loop

With the per-marker recipe established, the final step iterates over every entry in `geocoded_events` and applies the same construction. The loop variable `event` replaces the `sample_event` placeholder from the demonstration cells above. Every line in the loop body matches a line in the corresponding demonstration cell.

After the loop, the bare expression `m` on its own line causes Jupyter to render the interactive folium map inline as the cell's output. Omitting this final `m` line is a common pitfall — the loop runs successfully, the markers are attached, but no output appears and the cell looks broken.

In [ ]:
for event in geocoded_events:
    color = EVENT_COLORS.get(event.get("event_type"), "gray")
    popup_html = (
        f"<div style='width:260px'>"
        f"<b>{event['company']}</b><br>"
        f"<i>{event['file_date']} — {event['event_type']}</i><br>"
        f"{event['summary']}<br>"
        f"<a href='{event['url']}' target='_blank'>View SEC filing</a>"
        f"</div>"
    )
    folium.Marker(
        location=[event["latitude"], event["longitude"]],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=event["company"],
        icon=folium.Icon(color=color, icon="info-sign"),
    ).add_to(m)

m

> **💬 DISCUSSION PROMPT**
>
> Why did we map event types to colors using a dictionary instead of writing five separate `if` statements? What would the cost of the alternative approach be when the lesson is extended to fifteen event types instead of five?

## Evaluation notes

After your first end-to-end run, record the following observations. They serve as sanity checks and as the starting point for the assignment.

1. **Total filings returned by the keyword search** — captured in `total_found`. This is the upper bound on candidate volume for the chosen window.
2. **Fraction classified as genuine location events** — the ratio `events / fetched_filings`. The recall/precision discussion in Stage 3 explains why this percentage is typically low and why that is acceptable. If the rate is below approximately 8 percent, the keyword set is too broad and `"new location"` is the highest-leverage phrase to remove first.
3. **Geocoding success rate** — `geocoded_events / events`. Low rates typically indicate that the LLM is returning city names that Nominatim cannot resolve (often international locations, neighborhood-level descriptors, or generic regional language).
4. **Approximate cost per run** — visible in the Anthropic API console under Usage. With Haiku 4.5 at one dollar per million input tokens, a typical 100-filing run costs approximately thirty cents.

### Possible refinements for future iterations

- Pre-curate a list of ten to fifteen well-known retailers or logistics firms and restrict the EDGAR search by ticker. This produces a smaller, more reliable candidate set suitable for a single lab session.
- Replace the broad keyword query with item-code filtering (8-K Item 2.05 for exit activities, Item 2.01 for asset acquisitions). This sacrifices recall for precision.
- Add a second LLM call that produces a 100-word executive summary of the geographic pattern across all events, printed below the map.
- Cache results in a local CSV so you can iterate on the visualization without re-running the API pipeline.
- Add a date-range slider to the folium map so users can filter events by filing date interactively.

---

## Instructor reference: prototype validation numbers

These metrics were observed during prototype validation on May 3, 2026 (see `cis3120_edgar_folium_prototype_v2.ipynb`). They serve as expected ranges for sanity-checking that an in-class run is functioning correctly. If the class run produces numbers well outside these ranges, the most likely causes are noted below.

| Metric | Observed | Expected range | Most likely cause if outside range |
|:---|:---|:---|:---|
| EDGAR candidates (30-day window) | 111 | 90 to 130 | EDGAR rate-limited the requests; check User-Agent header |
| Successful exhibit fetches | 111 of 111 | ≥ 95% of candidates | Transient SEC errors; rerun the fetch loop |
| Filings classified as location events | 17 | 10 to 20 | Prompt has drifted; compare to canonical |
| Precision (verified) | 100% | ≥ 80% | Prompt is too permissive |
| Geocoded events | 16 of 17 | ≥ 90% of events | Nominatim was rate-limited; verify 1.1s sleep |
| Anthropic input tokens | 264,677 | 200,000 to 350,000 | Truncation cap was changed |
| Anthropic output tokens | 6,301 | 4,000 to 9,000 | Output schema changed |
| Cost per run (Haiku) | ~$0.30 | $0.25 to $0.40 | Wrong model selected |
| End-to-end runtime | ~4 minutes | 3 to 6 minutes | Network latency or rate-limiting |

For comparison, the same configuration on Opus 4.7 cost approximately $1.83 per run with 12 location events identified (lower recall) at the same precision. Haiku is unambiguously the right choice for this task.

## End-of-walkthrough discussion (5 minutes)

Before closing the session, surface the following themes briefly:

1. **The retrieval-then-classify pattern.** Stage 1 supplies recall, Stage 3 supplies precision. This division of labor is the pedagogical bridge to Module 16, where Stage 1 will be replaced by embedding-based retrieval over a corpus rather than keyword matching against EDGAR.
2. **Model selection as task fit.** The pipeline uses Haiku because the work is bounded classification and short-form structured extraction. Opus would be appropriate at points in a pipeline that genuinely require deep reasoning, such as multi-document synthesis or ambiguous classification — neither of which is present here.
3. **API composition discipline.** The pipeline coordinates three external services plus one client-side library. Each has its own rate limit, authentication requirement, and failure mode. The defensive patterns in this notebook (User-Agent headers, explicit sleep calls, retry-with-backoff for EDGAR, defensive JSON parsing for Claude output) generalize to every multi-service pipeline students will build going forward.

Close by previewing the assignment: students will extend this pipeline in one of three ways (time series across four weeks, sector comparison between retailers and manufacturers, or modification of the SEARCH_PHRASES list to target a different event category). Direct them to the assignment document for full requirements.